# TB Chest X-Ray Detection (DenseNet-121)

Fixed version — see inline notes for what changed from the original and why. Main fix: the training loader was pointed at the full dataset instead of the train split, which meant validation/test images were leaking into training.

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets as tv_datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
data_dir = "TB_Chest_Radiography_Database"

## Preprocessing & Augmentation

**Changed:**
- Normalization now uses ImageNet mean/std (what DenseNet-121's pretrained weights actually expect), not a generic `[0.5]/[0.5]`.
- Train transform now includes light augmentation (rotation, translation, mild brightness/contrast). No horizontal flip — chest X-ray left/right orientation is clinically meaningful, so flipping would create anatomically invalid images.
- Val/test transforms stay deterministic (resize + normalize only), since we want to evaluate on unaltered images.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Loading the data

**Changed:** renamed the dataset variable from `datasets` to `full_dataset` so it no longer shadows the `torchvision.datasets` module import.

We load the folder twice — once with `train_transform` (augmented) and once with `eval_transform` (clean) — then use the **same stratified split indices** on both, so train gets augmentation and val/test don't, without ever mixing samples across splits.

In [ ]:
full_dataset_train_tf = tv_datasets.ImageFolder(root=data_dir, transform=train_transform)
full_dataset_eval_tf = tv_datasets.ImageFolder(root=data_dir, transform=eval_transform)

class_names = full_dataset_train_tf.classes
labels = [label for _, label in full_dataset_train_tf.samples]
print("Classes:", class_names)

## Stratified split

**Changed:** the original used `torch.utils.data.random_split`, which doesn't preserve class balance across splits. With TB being the minority class here, an unlucky split could easily starve val/test of TB examples. We use a stratified split on indices instead, so each split keeps roughly the same Normal:TB ratio as the full dataset.

In [ ]:
indices = np.arange(len(full_dataset_train_tf))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=labels, random_state=SEED
)
temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_labels, random_state=SEED
)

train_dataset = Subset(full_dataset_train_tf, train_idx)   # augmented
val_dataset = Subset(full_dataset_eval_tf, val_idx)         # clean
test_dataset = Subset(full_dataset_eval_tf, test_idx)       # clean

total = len(full_dataset_train_tf)
print(f"Train: {len(train_dataset)} ({len(train_dataset)/total:.1%})")
print(f"Val:   {len(val_dataset)} ({len(val_dataset)/total:.1%})")
print(f"Test:  {len(test_dataset)} ({len(test_dataset)/total:.1%})")

**Changed (the main bug):** `train_loader` now points at `train_dataset` instead of the full dataset. In the original, `DataLoader(datasets, ...)` trained on every image — including the ones held out for validation and test — which is why the reported metrics looked close to perfect. That leakage is gone now.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## DenseNet-121

In [ ]:
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

In [ ]:
for param in model.parameters():
    param.requires_grad = False  # Freeze all layers by default

In [ ]:
# Unfreeze the last dense block and classifier for fine-tuning.
# This lets the model adapt high-level features to TB X-rays without
# destroying the low-level (edge/texture) features learned from ImageNet.
for param in model.features.denseblock4.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 2)  # binary: Normal vs Tuberculosis

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Using device:", device)

## Training

**Added:** basic checkpointing — we track the best validation loss and save that state dict, rather than whatever the model looked like after the last epoch. For a small dataset run over a few epochs this matters less, but it's good habit and costs nothing.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=0.001
)
num_epochs = 5

In [ ]:
best_val_loss = float("inf")
best_state_dict = None

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct, total_train = 0, 0, 0

    for images, labels_batch in train_loader:
        images, labels_batch = images.to(device), labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels_batch).sum().item()
        total_train += labels_batch.size(0)

    train_acc = train_correct / total_train
    train_loss = train_loss / total_train

    model.eval()
    val_loss, val_correct, total_val = 0, 0, 0
    with torch.no_grad():
        for images, labels_batch in val_loader:
            images, labels_batch = images.to(device), labels_batch.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels_batch)
            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels_batch).sum().item()
            total_val += labels_batch.size(0)

    val_acc = val_correct / total_val
    val_loss = val_loss / total_val

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# Load best checkpoint before evaluating / saving
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

## Evaluation

**Added:** per-class recall is called out explicitly (not just the classification report) since Tuberculosis is the minority class and recall on it is the number that matters most clinically — missing a real TB case is a much worse failure mode than a false alarm.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import matplotlib.pyplot as plt

model.eval()
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels_batch in test_loader:
        images, labels_batch = images.to(device), labels_batch.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(1)

        all_labels.extend(labels_batch.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

# ---- Classification report ----
print(classification_report(all_labels, all_preds, target_names=class_names))

# ---- Confusion matrix ----
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# ---- ROC curve and AUC ----
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - TB Detection')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# ---- Minority-class recall, called out explicitly ----
tb_index = class_names.index("Tuberculosis")
tb_recall = cm[tb_index, tb_index] / cm[tb_index].sum()
print(f"Tuberculosis recall (sensitivity): {tb_recall:.4f}")

## Save model

In [ ]:
torch.save(model.state_dict(), "tb_detector_densenet121.pth")
print("Model saved successfully!")

---

**Note:** this notebook hasn't been re-run here since the dataset isn't in this environment — run it against your local copy of `TB_Chest_Radiography_Database` to get real numbers. Expect the accuracy to come down from the original run (that's the leakage fix working as intended, not a regression) — the new numbers will just be trustworthy ones.